In [ ]:
#import sys
#import os

#sys.path.append(os.path.abspath(r"D:\projectpart1"))

import pandas as pd
from customer_cleaner import CustomerCleaner

df=pd.read_csv(r"D:\projectpart1\WA_Fn-UseC_-Telco-Customer-Churn.csv")

customclean=CustomerCleaner(df)
df=customclean.clean()

2026-08-07 10:53:32,897 - INFO - CustomerCleaner initialized with DataFrame shape (7043, 21)
2026-08-07 10:53:32,900 - INFO - Starting data cleaning pipeline
2026-08-07 10:53:32,902 - INFO - Standardizing column names
2026-08-07 10:53:32,907 - INFO - Column standardization completed
2026-08-07 10:53:32,908 - INFO - Fixing total_charges column
2026-08-07 10:53:32,924 - INFO - Converted total_charges to numeric. Blank rows affected: 11
2026-08-07 10:53:32,926 - INFO - Normalizing binary columns
2026-08-07 10:53:32,931 - INFO - Normalized column: partner
2026-08-07 10:53:32,937 - INFO - Normalized column: dependents
2026-08-07 10:53:32,943 - INFO - Normalized column: phone_service
2026-08-07 10:53:32,947 - INFO - Normalized column: paperless_billing
2026-08-07 10:53:32,950 - INFO - Normalized column: churn
2026-08-07 10:53:32,951 - INFO - Binary column normalization completed
2026-08-07 10:53:32,953 - INFO - Handling null values
2026-08-07 10:53:32,959 - INFO - Filled 11 null values in to

In [2]:
df.head()

,customer_id,gender,senior_citizen,partner,dependents,tenure,phone_service,multiple_lines,internet_service,online_security,...,device_protection,tech_support,streaming_tv,streaming_movies,contract,paperless_billing,payment_method,monthly_charges,total_charges,churn
0,7590-VHVEG,Female,0,1,0,1,0,No phone service,DSL,No,...,No,No,No,No,Month-to-month,1,Electronic check,29.85,29.85,0
1,5575-GNVDE,Male,0,0,0,34,1,No,DSL,Yes,...,Yes,No,No,No,One year,0,Mailed check,56.95,1889.50,0
2,3668-QPYBK,Male,0,0,0,2,1,No,DSL,Yes,...,No,No,No,No,Month-to-month,1,Mailed check,53.85,108.15,1
3,7795-CFOCW,Male,0,0,0,45,0,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,0,Bank transfer (automatic),42.30,1840.75,0
4,9237-HQITU,Female,0,0,0,2,1,No,Fiber optic,No,...,No,No,No,No,Month-to-month,1,Electronic check,70.70,151.65,1


In [ ]:
# 19. compute churn rate by contract type

round(df.groupby("contract")["churn"].mean() * 100, 2).sort_values(ascending=False)

contract
Month-to-month    42.71
One year          11.27
Two year           2.83
Name: churn, dtype: float64

In [ ]:
# 20. compute churn rate by internet service type

round(df.groupby("internet_service")["churn"].mean() * 100, 2).sort_values(ascending=False)

internet_service
Fiber optic    41.89
DSL            18.96
No              7.40
Name: churn, dtype: float64

In [ ]:
# 21. compute churn rate by payment method

round(df.groupby("payment_method")["churn"].mean() * 100, 2).sort_values(ascending=False)

payment_method
Electronic check             45.29
Mailed check                 19.11
Bank transfer (automatic)    16.71
Credit card (automatic)      15.24
Name: churn, dtype: float64

In [ ]:
# 22 create tenure bucket

bins=[0, 12, 24, 48, 72]
labels=["0-12", "13-24", "25-48", "49-72"]
df["tenure_bucket"]=pd.cut(
    df["tenure"],
    bins=bins,
    labels=labels,
    include_lowest=True
)

In [ ]:
# 23. Computer average on monthly_charges

round(df.groupby("churn")["monthly_charges"].mean().rename(index={0: "No", 1:"Yes"}),2)

churn
No     61.27
Yes    74.44
Name: monthly_charges, dtype: float64

In [ ]:
# 24. combination (contract + internet_service)

df.groupby(["contract", "internet_service"])["churn"].mean().sort_values(ascending=False)

contract        internet_service
Month-to-month  Fiber optic         0.546053
                DSL                 0.322159
One year        Fiber optic         0.192950
Month-to-month  No                  0.188931
One year        DSL                 0.092982
Two year        Fiber optic         0.072261
One year        No                  0.024725
Two year        DSL                 0.019108
                No                  0.007837
Name: churn, dtype: float64

In [ ]:
print(f"{"="*30} Profile Summary {"="*30}")

churn_rate=round(df.groupby("contract")["churn"].mean() * 100, 2).sort_values(ascending=False)
print(f"Churn rate table per contract ({churn_rate.idxmax()} will be highest, ~ {churn_rate.max()})")

churn_rate=round(df.groupby("tenure_bucket")["churn"].mean() * 100, 2).sort_values(ascending=False)
print(f"Churn rate table per contract ({churn_rate.idxmax()} will be highest, ~ {churn_rate.max()})")

churn_rate=round(df.groupby("payment_method")["churn"].mean() * 100, 2).sort_values(ascending=False)
print(f"Churn rate table per contract ({churn_rate.idxmax()} will be highest, ~ {churn_rate.max()})")

print("The retention team should focus on these customers first.")

============================== Profile Summary ==============================
Churn rate table per contract (Month-to-month will be highest, ~ 42.71)
Churn rate table per contract (0-12 will be highest, ~ 47.44)
Churn rate table per contract (Electronic check will be highest, ~ 45.29)
The retention team should focus on these customers first.


C:\Users\aravind.harish\AppData\Local\Temp\ipykernel_21376\1606386017.py:6: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  churn_rate = round(df.groupby("tenure_bucket")["churn"].mean() * 100, 2).sort_values(ascending=False)


In [ ]:
'''Start from the cleaned DataFrame (CustomerCleaner output) 

27.Feature 1 — tenure_bucket: pd.cut() with bins [0,12,24,48,72] as in CP3 

28.Feature 2 — high_charge_flag: 1 if monthly_charges > median monthly_charges, else 0 

29.Feature 3 — service_count: count how many of these 6 service columns equal 1: online_security, online_backup, device_protection, tech_support, streaming_tv, streaming_movies 

30.Feature 4 — is_long_term_customer: 1 if tenure >=24, else 0 

31.Feature 5 — has_streaming_bundle: 1 if both streaming_tv=1 AND streaming_movies=1, else 0 

32.Feature 6 — auto_pay_flag: 1 if payment_method contains "automatic", else 0 (use str.contains()) 

33.Print the correlation of each new feature with the churn column 

34.Save to customer_features.csv — include all original cleaned columns plus the 6 new ones '''

'Start from the cleaned DataFrame (CustomerCleaner output) \n\n27.Feature 1 — tenure_bucket: pd.cut() with bins [0,12,24,48,72] as in CP3 \n\n28.Feature 2 — high_charge_flag: 1 if monthly_charges > median monthly_charges, else 0 \n\n29.Feature 3 — service_count: count how many of these 6 service columns equal 1: online_security, online_backup, device_protection, tech_support, streaming_tv, streaming_movies \n\n30.Feature 4 — is_long_term_customer: 1 if tenure >= 24, else 0 \n\n31.Feature 5 — has_streaming_bundle: 1 if both streaming_tv=1 AND streaming_movies=1, else 0 \n\n32.Feature 6 — auto_pay_flag: 1 if payment_method contains "automatic", else 0 (use str.contains()) \n\n33.Print the correlation of each new feature with the churn column \n\n34.Save to customer_features.csv — include all original cleaned columns plus the 6 new ones '

In [ ]:
#27.Feature 1 — tenure_bucket: pd.cut() with bins [0,12,24,48,72] as in CP3 
bins=[0,12,24,48,72]
labels=["0-12","13-24","25-48","49-72"]
df["tenure_bucket"]=pd.cut(df["tenure"],bins=bins,labels=labels)


In [ ]:
#28.Feature 2 — high_charge_flag: 1 if monthly_charges > median monthly_charges, else 0
median_monthly_charges=df["monthly_charges"].median()
df["high_charge_flag"]=(df["monthly_charges"] > median_monthly_charges).astype(int)
print(df["high_charge_flag"].value_counts())

high_charge_flag
0    3528
1    3515
Name: count, dtype: int64


In [ ]:
#29.Feature 3 — service_count: count how many of these 6 service columns equal 1: online_security, online_backup, device_protection, tech_support, streaming_tv, streaming_movies

service_columns=[
    "online_security",
    "online_backup",
    "device_protection",
    "tech_support",
    "streaming_tv",
    "streaming_movies"
]

mapping={"Yes": 1, "No": 0}

for col in service_columns:
    df[col]=df[col].replace(mapping).fillna(0).astype(int)

df["service_count"]=df[service_columns].sum(axis=1)
print(df[service_columns].head())


   online_security  online_backup  device_protection  tech_support  \
0                0              1                  0             0   
1                1              0                  1             0   
2                1              1                  0             0   
3                1              0                  1             1   
4                0              0                  0             0   

   streaming_tv  streaming_movies  
0             0                 0  
1             0                 0  
2             0                 0  
3             0                 0  
4             0                 0  


C:\Users\aravind.harish\AppData\Local\Temp\ipykernel_21376\472357931.py:15: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[col] = df[col].replace(mapping).fillna(0).astype(int)
C:\Users\aravind.harish\AppData\Local\Temp\ipykernel_21376\472357931.py:15: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[col] = df[col].replace(mapping).fillna(0).astype(int)
C:\Users\aravind.harish\AppData\Local\Temp\ipykernel_21376\472357931.py:15: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version

In [ ]:
#30.Feature 4 — is_long_term_customer: 1 if tenure >=24, else 0
df["is_long_term_customer"]=(df["tenure"] >=24).astype(int)
print(df["is_long_term_customer"].value_counts())


is_long_term_customer
1    3927
0    3116
Name: count, dtype: int64


In [ ]:
#31.Feature 5 — has_streaming_bundle: 1 if both streaming_tv=1 AND streaming_movies=1, else 0
#31.Feature 5 — has_streaming_bundle: 1 if both streaming_tv=1 AND streaming_movies=1, else 0
df["has_streaming_bundle"]=(
    ((df["streaming_tv"]==1) & (df["streaming_movies"]==1)).astype(int)
)
print(df["has_streaming_bundle"].value_counts())


has_streaming_bundle
0    5103
1    1940
Name: count, dtype: int64


In [ ]:
#32.Feature 6 — auto_pay_flag: 1 if payment_method contains "automatic", else 0 (use str.contains())
df["auto_pay_flag"]=(
    df["payment_method"]
    .astype(str)
    .str.contains("automatic", case=False, na=False)
    .astype(int)
)
print(df["auto_pay_flag"].value_counts())

auto_pay_flag
0    3977
1    3066
Name: count, dtype: int64


In [ ]:
#33.Print the correlation of each new feature with the churn column 

corr_features=df[
    ["high_charge_flag",
        "service_count",
        "is_long_term_customer",
        "has_streaming_bundle",
        "auto_pay_flag",
        "churn",
    ]
].corr()["churn"]

tenure_bucket_corr=(
    df["tenure_bucket"]
    .astype("category")
    .cat.codes
    .corr(df["churn"])
)

print("tenure_bucket vs churn correlation:", round(tenure_bucket_corr, 4))
print("other new feature correlations with churn:")
print(round(corr_features.drop("churn"), 4))



tenure_bucket vs churn correlation: -0.3439
other new feature correlations with churn:
high_charge_flag         0.1957
service_count           -0.0877
is_long_term_customer   -0.3115
has_streaming_bundle     0.0404
auto_pay_flag           -0.2099
Name: churn, dtype: float64


In [ ]:
#34.Save to customer_features.csv — include all original cleaned columns plus the 6 new ones
df.to_csv('customer_features.csv', index=False)

In [ ]:
'''35.Create customer_pipeline.py 

36.Import CustomerCleaner from customer_cleaner.py and build_features() logic 

37.Define load_data(filepath) — reads CSV and logs row count 

38.Define build_features(df) — runs all 6 feature engineering steps from CP4 and returns the enriched DataFrame 

39.Define save_outputs(clean_df, feature_df, output_dir) — saves both files with timestamps in filenames 

40.Define main() — calls load → clean → build_features → save in sequence 

41.Add logging throughout: start time, step completion, row counts, output file paths 

42.Wrap each step in try/except — if any step fails, log the error and raise; do not silently continue 

43.Add a simple data quality check before saving: assert no nulls in monthly_charges, assert churn column has only 0/1 values 

44.Run the script end-to-end: python customer_pipeline.py'''